In [ ]:
#notebook for testing slicegpt on gemma3

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# CHANGE this to your repo folder
%cd /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications
!ls


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications
build_and_test.sh   LICENSE	    README.md	 SUPPORT.md
CODE_OF_CONDUCT.md  pipelines	    SECURITY.md  tests
experiments	    pyproject.toml  src		 test.sh


In [ ]:
!pip install -e ".[experiment]"

In [ ]:
import slicegpt
from slicegpt import rotate, model_utils
print("slicegpt imported:", slicegpt.__file__)
print("rotate imported:", rotate.__file__)
print("model_utils imported:", model_utils.__file__)

slicegpt imported: /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/src/slicegpt/__init__.py
rotate imported: /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/src/slicegpt/rotate.py
model_utils imported: /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/src/slicegpt/model_utils.py


In [ ]:
import os, textwrap

# Where to save logs and models in your Drive
BASE_RESULTS_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments"
LOG_DIR = os.path.join(BASE_RESULTS_DIR, "logs_gemma3")
MODEL_DIR = os.path.join(BASE_RESULTS_DIR, "models_gemma3")
BASE_EVAL_DIR = os.path.join(BASE_RESULTS_DIR, "eval")

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

#sparsities = [0.0, 0.1, 0.25, 0.4, 0.6]
sparsities = [0.15]
datasets = ["squad"]

print("Logs in:", LOG_DIR)
print("Models in:", MODEL_DIR)

Logs in: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/logs_opt125M
Models in: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_opt125M


In [ ]:
from huggingface_hub import login
from google.colab import userdata
login(userdata.get("HF_TOKEN"))

In [ ]:
!pip uninstall -y transformers tokenizers
!pip install -U "transformers>=4.50.0" tokenizers accelerate safetensors
!python -c "import transformers; print(transformers.__version__)"

In [ ]:
# Cell 1: Slicing with correct dtype
import os
import subprocess
from datetime import datetime

MODEL_ID = "google/gemma-3-270m"

def run_slicegpt(dataset, sparsity):
    log_name = f"{MODEL_ID.replace('/','-')}_{dataset}_s{sparsity:.2f}.txt".replace(".", "p")
    log_path = os.path.join(LOG_DIR, log_name)

    if os.path.exists(log_path):
        print(f"[SKIP] Log already exists for {dataset}, sparsity={sparsity}: {log_path}")
        return

    save_dir = os.path.join(MODEL_DIR, f"{MODEL_ID.replace('/','-')}_{dataset}_s{sparsity:.2f}".replace(".", "p"))
    os.makedirs(save_dir, exist_ok=True)

    cmd = [
        "python",
        "/content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/experiments/run_slicegpt.py",
        "--model", MODEL_ID,
        "--cal-dataset", dataset,
        "--save-dir", save_dir,
        "--sparsity", str(sparsity),
        "--device", "cuda:0",
        "--no-wandb",
        "--cal-batch-size", "8"
        # Removed --dtype flag - will use model's default (fp16 for Gemma)
    ]

    print("\n=====================================================")
    print("Running:", " ".join(cmd))
    print("Log file:", log_path)
    print("Start:", datetime.now())
    print("=====================================================\n")

    with open(log_path, "w") as f:
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="")
            f.write(line)

    ret = process.wait()
    print("\nFinished with return code:", ret)
    print("End:", datetime.now())

for dataset in datasets:
    for s in sparsities:
        run_slicegpt(dataset, s)

In [ ]:
# Cell 1: Slicing with correct dtype
import os
import subprocess
from datetime import datetime

MODEL_ID = "google/gemma-3-270m"

def run_slicegpt(dataset, sparsity):
    log_name = f"{MODEL_ID.replace('/','-')}_{dataset}_s{sparsity:.2f}.txt".replace(".", "p")
    log_path = os.path.join(LOG_DIR, log_name)

    #if os.path.exists(log_path):
        #print(f"[SKIP] Log already exists for {dataset}, sparsity={sparsity}: {log_path}")
        #return

    save_dir = os.path.join(MODEL_DIR, f"{MODEL_ID.replace('/','-')}_{dataset}_s{sparsity:.2f}".replace(".", "p"))
    os.makedirs(save_dir, exist_ok=True)

    cmd = [
        "python",
        "/content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/experiments/run_slicegpt.py",
        "--model", MODEL_ID,
        "--cal-dataset", dataset,
        "--save-dir", save_dir,
        "--sparsity", str(sparsity),
        "--device", "cuda:0",
        "--no-wandb",
        "--cal-batch-size", "8"
        # Removed --dtype flag - will use model's default (fp16 for Gemma)
    ]

    print("\n=====================================================")
    print("Running:", " ".join(cmd))
    print("Log file:", log_path)
    print("Start:", datetime.now())
    print("=====================================================\n")

    with open(log_path, "w", buffering=1) as f:  # Line buffering for immediate writes
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="")
            f.write(line)
            f.flush()  # Ensure immediate write to disk

    ret = process.wait()
    print("\nFinished with return code:", ret)
    print("End:", datetime.now())

for dataset in datasets:
    for s in sparsities:
        run_slicegpt(dataset, s)


Running: python /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/experiments/run_slicegpt.py --model google/gemma-3-270m --cal-dataset squad --save-dir /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_gemma3/google-gemma-3-270m_squad_s0p15 --sparsity 0.15 --device cuda:0 --no-wandb --cal-batch-size 8
Log file: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/logs_gemma3/google-gemma-3-270m_squad_s0p15ptxt
Start: 2026-01-08 23:27:18.023994

2026-01-08 23:27:24.441040: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767914844.462406   10466 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767914844.468857   10466 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS

In [ ]:
import logging
import sys

# Get the root logger or a named logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)  # allow INFO and above

for h in list(logger.handlers):
  logger.removeHandler(h)

# Create a handler that writes to stdout
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)

# (Optional) set a formatting for readability
formatter = logging.Formatter('%(levelname)s - %(message)s')
handler.setFormatter(formatter)

# Add handler to the logger
logger.addHandler(handler)

# Now test
logger.info("This will be printed to stdout")
logger.debug("This will not print (level is INFO)")

INFO - This will be printed to stdout


In [ ]:
#fixed eval cell
import os
import sys
import json
import logging

import lm_eval
from lm_eval import tasks
from lm_eval import utils as lm_eval_utils
from lm_eval.api.registry import ALL_TASKS
from lm_eval.models.huggingface import HFLM

from slicegpt import gpu_utils, hf_utils, utils
from slicegpt.config import config

# Do NOT reconfigure logging here if you've already done it in a previous cell.
# We just reuse the existing logger.
logger = logging.getLogger(__name__)


def eval(args):
    """
    Run LM Evaluation Harness on a sliced (pruned) model.

    Expected args (dict-like):
      - "model": HF model name, e.g. "facebook/opt-125m"
      - "sliced_model_path": path to sliced checkpoint dir
      - "sparsity": float, e.g. 0.25
      - "round_interval": int or None (optional)
      - "batch_size": int
      - "tasks": None, string (comma-separated) or list of patterns
      - "num_fewshot": int
      - "limit": int or None
      - "save_dir": output directory for results
    """

    logger.info("Running Evaluation")

    # --- Load sliced model ---
    logger.info(
        f"Loading sliced {args['model']} from {args['sliced_model_path']} "
        f"with sparsity {args['sparsity']}"
    )

    model_adapter, tokenizer = hf_utils.load_sliced_model(
        args["model"],
        args["sliced_model_path"],
        sparsity=args["sparsity"],
        token=None,
        round_interval=args.get("round_interval", None),
    )

    # The lm-eval harness normally ties weights; we disable this for sliced models
    if hasattr(model_adapter.model, "tie_weights"):
        model_adapter.model.tie_weights = lambda *x, **kw: None

    model_adapter.model.to(config.device)

    # Wrap in LM Evaluation Harness HF adapter
    hflm = HFLM(
        pretrained=model_adapter.model,
        tokenizer=tokenizer,
        batch_size=args["batch_size"],
    )

    # --- Task selection ---
    if args["tasks"] is None:
        # Using ALL_TASKS is usually too heavy for Colab; you can change this to raise if you prefer.
        logger.warning(
            "args['tasks'] is None -> using ALL_TASKS. "
            "This may be very slow / memory-heavy."
        )
        task_names = tasks.ALL_TASKS
    else:
        # Accept either a comma-separated string or a list
        if isinstance(args["tasks"], str):
            patterns = [t.strip() for t in args["tasks"].split(",") if t.strip()]
        else:
            patterns = args["tasks"]
        task_names = lm_eval_utils.pattern_match(patterns, ALL_TASKS)

    logger.info(f"Selected Tasks: {task_names}")

    # --- Run evaluation ---
    results = lm_eval.simple_evaluate(
        hflm,
        tasks=task_names,
        num_fewshot=args["num_fewshot"],
        batch_size=args["batch_size"],
        limit=args["limit"],
        write_out=False,
        log_samples=False,
    )

    results = results["results"]
    #logger.info("Results:")
    #logger.info(results)

    # --- Save results ---
    os.makedirs(args["save_dir"], exist_ok=True)

    sparsity_tag = f"{args['sparsity']:.2f}"  # e.g. "0.25"
    # Optional: sanitize model name if you want to include it
    # model_name = args["model"].replace("/", "-")

    result_path = os.path.join(
        args["save_dir"],
        f"results_s{sparsity_tag}_{'_'.join(task_names)}.json",
    )

    with open(result_path, "w") as f:
        json.dump(results, f, indent=2)

    logger.info(f"Saved results to {result_path}")

    return results


INFO - NumExpr defaulting to 12 threads.
INFO - PyTorch version 2.9.0+cu126 available.
INFO - TensorFlow version 2.19.0 available.
INFO - JAX version 0.7.2 available.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# Cell 3: Run evaluation
model = "facebook/opt-125m"
model_name = model.replace("/", "-")
datasets = ['squad']
sparsities = [0.15]

dir_prefix = "squadEval"
MODEL_DIR = os.path.join(BASE_RESULTS_DIR, "models_opt125M")

results = None

for dataset in datasets:
    for sparsity in sparsities:
        EVAL_DIR = os.path.join(
            BASE_EVAL_DIR,
            dir_prefix,
            f"{model_name}_{dataset}"
        )
        os.makedirs(EVAL_DIR, exist_ok=True)

        args = {
            "model": model,
            "sliced_model_path": os.path.join(MODEL_DIR, f"{model_name}_{dataset}_s{sparsity:.2f}".replace(".", "p")),
            "sparsity": sparsity,
            "save_dir": EVAL_DIR,
            "tasks": ["squadv2"],
            "num_fewshot": 0,
            "batch_size": 8,
            "round_interval": 8,
            "limit": 100,  # ← Changed from 10 (still small but more representative)
            "max_input_len": 512,
            #"max_gen_toks": 16,  # ← Added explicit generation limit
        }

        print(f"\n{'='*60}")
        print(f"Evaluating: {model_name} on {dataset} with sparsity {sparsity}")
        print(f"{'='*60}\n")

        results = eval(args)


Evaluating: facebook-opt-125m on squad with sparsity 0.15

INFO - Running Evaluation
INFO - Loading sliced facebook/opt-125m from /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_opt125M/squad_s0p15 with sparsity 0.15
INFO - Loading facebook/opt-125m config  from Hugging Face


`torch_dtype` is deprecated! Use `dtype` instead!


INFO - Loading model done
INFO - Replacing layers


Instantiating OPTAttention without passing a `layer_idx` is not recommended and will lead to errors during the forward call if caching is used. Please make sure to provide a `layer_idx` when creating this class.


INFO - Replacing layers done
INFO - Fusing layernorm modules
INFO - Fusing layernorm modules done
INFO - Inferred target dimension from final layer 11 MLP output: 768
INFO - Inferred embedding dimension from first layer input: 648
INFO - Loading sliced model weights from /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_opt125M/squad_s0p15
INFO - Using encoder output dimension from config: 768
INFO - Cross-attention K/V will be sliced to encoder output dim: 768
INFO - ✓ Loaded sliced model with learned rotation shortcuts
WARNING - `pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
WARNING - Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration
INFO - Selected Tasks: ['squadv2']
INFO - Building contexts for task on rank 0...
INFO - Running generate_until re

  0%|          | 0/100 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1786: FutureWarning: `past_key_value` is deprecated and will be removed in version 4.58 for `OPTAttention.forward`. Use `past_key_values` instead.
  return forward_call(*args, **kwargs)
100%|██████████| 100/100 [00:49<00:00,  2.03it/s]

INFO - Running loglikelihood requests



100%|██████████| 100/100 [00:00<00:00, 305.40it/s]
/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/squadv2/task.py:40: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  squad_metric = datasets.load_metric("squad_v2")
/usr/local/lib/python3.12/dist-packages/datasets/load.py:756: FutureWarning: The repository for squad_v2 contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.18.0/metrics/squad_v2/squad_v2.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(


/usr/local/lib/python3.12/dist-packages/datasets/load.py:756: FutureWarning: The repository for squad_v2 contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.18.0/metrics/squad_v2/squad_v2.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:756: FutureWarning: The repository for squad_v2 contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.18.0/metrics/squad_v2/squad_v2.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric 

INFO - Saved results to /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squadEval/facebook-opt-125m_squad/results_s0.15_squadv2.json


In [ ]:
# Cell 4: Quick diagnostic test (run BEFORE evaluation)
import torch
from slicegpt import hf_utils

print("Running diagnostic test...")

model_adapter, tokenizer = hf_utils.load_sliced_model(
    "google/gemma-3-270m",
    "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_gemma3/google-gemma-3-270m_squad_s0p10",
    sparsity=0.10
)

model = model_adapter.model.to(device="cuda", dtype=torch.float16).eval()

# Check 1: Model architecture (Gemma is decoder-only, no encoder/decoder split)
print("\n" + "="*60)
print("CHECK 1: Model Architecture")
print("="*60)
print(f"✓ Model type: {type(model).__name__}")
print(f"  Embedding shape: {model.model.embed_tokens.weight.shape}")
print(f"  LM head shape: {model.lm_head.weight.shape}")
print(f"  Number of layers: {len(model.model.layers)}")

# Check 2: Shortcut dimensions
print("\n" + "="*60)
print("CHECK 2: Shortcut Dimensions")
print("="*60)
for i in range(len(model.model.layers)):
    layer = model.model.layers[i]
    attn_shape = layer.attn_shortcut_Q.shape if hasattr(layer, 'attn_shortcut_Q') else None
    mlp_shape = layer.mlp_shortcut_Q.shape if hasattr(layer, 'mlp_shortcut_Q') else None

    # Only print layers with non-square shortcuts or every 5th layer
    if (attn_shape and attn_shape[0] != attn_shape[1]) or (mlp_shape and mlp_shape[0] != mlp_shape[1]) or i % 5 == 0 or i == 17:
        print(f"  Layer {i:2d}: attn_shortcut={attn_shape}, mlp_shortcut={mlp_shape}")

# Check 3: NaN/Inf
print("\n" + "="*60)
print("CHECK 3: Parameter Sanity")
print("="*60)
has_nan = False
for name, param in model.named_parameters():
    if not torch.isfinite(param).all():
        print(f"✗ Non-finite values in {name}")
        has_nan = True
        break
if not has_nan:
    print("✓ All parameters are finite")

# Check 4: Forward pass
print("\n" + "="*60)
print("CHECK 4: Forward Pass")
print("="*60)
prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    try:
        # Gemma doesn't need labels for forward pass
        out = model(**inputs)
        logits_finite = torch.isfinite(out.logits).all().item()
        logits_mean = out.logits.mean().item()
        logits_std = out.logits.std().item()

        print(f"✓ Logits finite: {logits_finite}" if logits_finite else f"✗ Logits have NaN/Inf")
        print(f"  Logits mean: {logits_mean:.4f}, std: {logits_std:.4f}")
    except Exception as e:
        print(f"✗ Forward pass failed: {e}")
        import traceback
        traceback.print_exc()

# Check 5: Generation
print("\n" + "="*60)
print("CHECK 5: Generation Quality")
print("="*60)
test_prompts = [
    "The capital of France is",
    "2 + 2 =",
    "Question: What is the largest planet? Answer:"
]

all_good = True
for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        try:
            gen = model.generate(
                **inputs,
                max_new_tokens=16,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
            text = tokenizer.decode(gen[0], skip_special_tokens=True)

            # Check if it's repetitive garbage
            tokens = text.split()
            unique_ratio = len(set(tokens)) / max(len(tokens), 1)
            is_garbage = unique_ratio < 0.3 or len(tokens) < 2
            status = "✗ GARBAGE" if is_garbage else "✓ OK"

            print(f"\n{status} (unique_ratio: {unique_ratio:.2f})")
            print(f"  Prompt: {prompt}")
            print(f"  Output: {repr(text[:200])}")

            if is_garbage:
                all_good = False
        except Exception as e:
            print(f"✗ Generation failed: {e}")
            all_good = False

print("\n" + "="*60)
if all_good and not has_nan:
    print("✓✓✓ ALL CHECKS PASSED - Model appears healthy!")
else:
    print("✗✗✗ CHECKS FAILED - Model may be broken, review output above")
print("="*60)

Running diagnostic test...
INFO - Loading google/gemma-3-270m config  from Hugging Face
INFO - Loading model done
INFO - Replacing layers
INFO - Replacing layers done
INFO - Fusing layernorm modules
INFO - Fusing layernorm modules done
INFO - Inferred target dimension from final layer 17 MLP output: 640
INFO - Inferred embedding dimension from first layer input: 576
INFO - Loading sliced model weights from /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_gemma3/google-gemma-3-270m_squad_s0p10
INFO - Using encoder output dimension from config: 640
INFO - Cross-attention K/V will be sliced to encoder output dim: 640

CHECK 1: Model Architecture
✓ Model type: UninitializedGemma3ForCausalLM
  Embedding shape: torch.Size([262144, 576])
  LM head shape: torch.Size([262144, 640])
  Number of layers: 18

CHECK 2: Shortcut Dimensions
  Layer  0: attn_shortcut=torch.Size([576, 576]), mlp_shortcut=torch.Size([576, 576])
  Layer  5: attn_shortcut=torch.Size([576, 576]), mlp_shortcut=

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1786: FutureWarning: Both `past_key_value` and `past_key_values` are set for `Gemma3Attention.forward`. Using `past_key_values=None` and ignoring deprecated `past_key_value=None`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1786: FutureWarning: Both `past_key_value` and `past_key_values` are set for `Gemma3Attention.forward`. Using `past_key_values=DynamicCache(layers=[DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicSlidingWindowLayer, DynamicLayer])` and ignoring deprecated `past_key_value=None`.
  r


✓ OK (unique_ratio: 1.00)
  Prompt: The capital of France is
  Output: 'The capital of France is'

✓ OK (unique_ratio: 0.75)
  Prompt: 2 + 2 =
  Output: '2 + 2 ='

✓ OK (unique_ratio: 1.00)
  Prompt: Question: What is the largest planet? Answer:
  Output: 'Question: What is the largest planet? Answer:'

✓✓✓ ALL CHECKS PASSED - Model appears healthy!


In [ ]:
# Fixed enhanced diagnostic test
import torch
from slicegpt import hf_utils

print("Running enhanced diagnostic test...")

model_adapter, tokenizer = hf_utils.load_sliced_model(
    "google/gemma-3-270m",
    "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_gemma3/google-gemma-3-270m_squad_s0p15",
    sparsity=0.15
)

model = model_adapter.model.to(device="cuda", dtype=torch.float16).eval()

print("\n" + "="*60)
print("SHORTCUT INSPECTION")
print("="*60)

for i in [0, 5, 10, 15, 16, 17]:
    layer = model.model.layers[i]

    attn_shape = layer.attn_shortcut_Q.shape
    mlp_shape = layer.mlp_shortcut_Q.shape

    # Check if shortcuts are identity-like
    attn_is_square = attn_shape[0] == attn_shape[1]
    mlp_is_square = mlp_shape[0] == mlp_shape[1]

    print(f"\nLayer {i:2d}:")
    print(f"  attn_shortcut: shape={attn_shape}, square={attn_is_square}")
    print(f"  mlp_shortcut:  shape={mlp_shape}, square={mlp_is_square}")

    # Check diagonal sum for identity check
    if attn_is_square:
        diag_sum = torch.diagonal(layer.attn_shortcut_Q).sum().item()
        expected = attn_shape[0]
        print(f"    attn diagonal sum: {diag_sum:.1f} (expected: {expected})")

    if mlp_is_square:
        diag_sum = torch.diagonal(layer.mlp_shortcut_Q).sum().item()
        expected = mlp_shape[0]
        print(f"    mlp diagonal sum: {diag_sum:.1f} (expected: {expected})")
    else:
        # Rectangular - check the identity part
        min_dim = min(mlp_shape)
        diag_sum = torch.diagonal(layer.mlp_shortcut_Q[:min_dim, :min_dim]).sum().item()
        print(f"    mlp diagonal sum (first {min_dim}×{min_dim}): {diag_sum:.1f} (expected: {min_dim})")

        # Check if padding is zeros
        if mlp_shape[0] < mlp_shape[1]:
            # Expanded (576→640): check right padding
            padding = layer.mlp_shortcut_Q[:, min_dim:]
            padding_sum = padding.abs().sum().item()
            print(f"    mlp padding (cols {min_dim}:{mlp_shape[1]}): sum={padding_sum:.6f} (should be ~0)")

    # Check for NaN/Inf in shortcuts
    attn_finite = torch.isfinite(layer.attn_shortcut_Q).all().item()
    mlp_finite = torch.isfinite(layer.mlp_shortcut_Q).all().item()
    print(f"  Finite: attn={attn_finite}, mlp={mlp_finite}")

print("\n" + "="*60)
print("FORWARD PASS WITH HOOKS")
print("="*60)

# Add hooks to track NaN propagation
nan_layer = None

def make_hook(layer_idx):
    def hook(module, input, output):
        global nan_layer
        if isinstance(output, tuple):
            out_tensor = output[0]
        else:
            out_tensor = output

        if not torch.isfinite(out_tensor).all():
            if nan_layer is None:
                nan_layer = layer_idx
                print(f"\n  ⚠️  FIRST NaN detected at layer {layer_idx}")
                print(f"      Output shape: {out_tensor.shape}")
                print(f"      Output mean: {out_tensor.mean().item() if torch.isfinite(out_tensor.mean()) else 'NaN'}")
    return hook

# Register hooks
handles = []
for i, layer in enumerate(model.model.layers):
    handle = layer.register_forward_hook(make_hook(i))
    handles.append(handle)

# Run forward pass
prompt = "Hello"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    try:
        out = model(**inputs)
        logits_finite = torch.isfinite(out.logits).all().item()

        if logits_finite:
            print("✓ Forward pass completed successfully - no NaN detected")
            print(f"  Logits: mean={out.logits.mean().item():.4f}, std={out.logits.std().item():.4f}")
        else:
            print(f"✗ Logits contain NaN (first NaN was at layer {nan_layer})")
    except Exception as e:
        print(f"✗ Forward pass failed with error: {e}")
        import traceback
        traceback.print_exc()

# Remove hooks
for handle in handles:
    handle.remove()

print("\n" + "="*60)
print("TESTING FLOAT32 vs FLOAT16")
print("="*60)

# Try float32
print("\nTesting with float32...")
model_fp32 = model_adapter.model.to(device="cuda", dtype=torch.float32).eval()

with torch.no_grad():
    out_fp32 = model_fp32(**inputs)
    logits_finite_fp32 = torch.isfinite(out_fp32.logits).all().item()

    if logits_finite_fp32:
        print(f"✓ FP32: Logits are finite")
        print(f"  Mean: {out_fp32.logits.mean().item():.4f}, Std: {out_fp32.logits.std().item():.4f}")
    else:
        print(f"✗ FP32: Logits contain NaN")

print("\n" + "="*60)
print("GENERATION TEST")
print("="*60)

if logits_finite_fp32:
    print("\nTrying generation with FP32 model...")
    test_prompt = "The capital of France is"
    inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        gen = model_fp32.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            num_beams=1
        )
        output_text = tokenizer.decode(gen[0], skip_special_tokens=True)

        # Check if it generated new tokens
        input_text = tokenizer.decode(inputs.input_ids[0], skip_special_tokens=True)
        generated_new = len(output_text) > len(input_text)

        print(f"  Input:  '{input_text}'")
        print(f"  Output: '{output_text}'")
        print(f"  Generated new tokens: {generated_new}")


INFO - NumExpr defaulting to 12 threads.
INFO - PyTorch version 2.9.0+cu126 available.
INFO - TensorFlow version 2.19.0 available.
INFO - JAX version 0.7.2 available.
Running enhanced diagnostic test...
INFO - Loading google/gemma-3-270m config  from Hugging Face


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


INFO - Loading model done
INFO - Replacing layers
INFO - Replacing layers done
INFO - Fusing layernorm modules
INFO - Fusing layernorm modules done
INFO - Inferred target dimension from final layer 17 MLP output: 640
INFO - Inferred embedding dimension from first layer input: 544


TypeError: 'NoneType' object is not subscriptable

In [ ]:
import torch
from slicegpt import hf_utils

model_adapter, tokenizer = hf_utils.load_sliced_model(
    "google/gemma-3-270m",
    "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_gemma3/google-gemma-3-270m_squad_s0p10",
    sparsity=0.10
)

model = model_adapter.model.to(device="cuda", dtype=torch.float16).eval()

print("Checking dtypes...")
for i in [0, 11, 17]:
    layer = model.model.layers[i]
    print(f"\nLayer {i}:")
    print(f"  self_attn.q_proj.weight dtype: {layer.self_attn.q_proj.weight.dtype}")
    print(f"  attn_shortcut_Q dtype: {layer.attn_shortcut_Q.dtype}")
    print(f"  mlp_shortcut_Q dtype: {layer.mlp_shortcut_Q.dtype}")


INFO - Loading google/gemma-3-270m config  from Hugging Face
INFO - Loading model done
INFO - Replacing layers
INFO - Replacing layers done
INFO - Fusing layernorm modules
INFO - Fusing layernorm modules done
INFO - Inferred target dimension from final layer 17 MLP output: 640
INFO - Inferred embedding dimension from first layer input: 576
INFO - Loading sliced model weights from /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_gemma3/google-gemma-3-270m_squad_s0p10
INFO - Using encoder output dimension from config: 640
INFO - Cross-attention K/V will be sliced to encoder output dim: 640
Checking dtypes...

Layer 0:
  self_attn.q_proj.weight dtype: torch.float16
  attn_shortcut_Q dtype: torch.float16
  mlp_shortcut_Q dtype: torch.float16

Layer 11:
  self_attn.q_proj.weight dtype: torch.float16
  attn_shortcut_Q dtype: torch.float16
  mlp_shortcut_Q dtype: torch.float16

Layer 17:
  self_attn.q_proj.weight dtype: torch.float16
  attn_shortcut_Q dtype: torch.float16
  mlp_

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load original model
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-270m")
model = AutoModelForCausalLM.from_pretrained("google/gemma-3-270m", torch_dtype=torch.float32).to("cuda")

prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    gen = model.generate(**inputs, max_new_tokens=10, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    output = tokenizer.decode(gen[0], skip_special_tokens=True)

print(f"Input:  '{prompt}'")
print(f"Output: '{output}'")


The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Input:  'The capital of France is'
Output: 'The capital of France is Paris. It is the most visited city in the'


In [ ]:
import os

checkpoint_path = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_gemma3/google-gemma-3-270m_squad_s0p10/gemma-3-270m_0.1.pt"

if os.path.exists(checkpoint_path):
    size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)
    # Get file modification time
    import datetime
    mtime = os.path.getmtime(checkpoint_path)
    mod_time = datetime.datetime.fromtimestamp(mtime)
    print(f"✓ Checkpoint exists: {size_mb:.2f} MB")
    print(f"  Last modified: {mod_time}")
else:
    print("✗ Checkpoint does NOT exist!")


✓ Checkpoint exists: 1633.31 MB
  Last modified: 2026-01-06 22:48:43
